#### Imports and setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import RAW_DATA_FILE, TARGET_COLUMN
from src.data_loader import load_energy_dataset, parse_datetime_column
from src.feature_engineering_utils import (
    clip_outliers_iqr,
    select_columns,
    create_time_features,
    create_lag_features,
    create_rolling_features,
    create_interaction_features,
    drop_feature_engineering_nulls,
    save_dataframe,
)

pd.set_option("display.max_columns", None)

## Feature Engineering

This notebook creates additional predictive features for the appliance energy forecasting task.  
The engineered features include time-based features, lagged features, rolling statistics, and interaction features.

#### 1) Load Dataset

In [2]:
raw_df = load_energy_dataset(RAW_DATA_FILE)
raw_df = parse_datetime_column(raw_df, datetime_column="date", set_as_index=True)

raw_df.head()

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,RH_5,T6,RH_6,T7,RH_7,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,45.566667,17.166667,55.20,7.026667,84.256667,17.200000,41.626667,18.2,48.900000,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,45.992500,17.166667,55.20,6.833333,84.063333,17.200000,41.560000,18.2,48.863333,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,45.890000,17.166667,55.09,6.560000,83.156667,17.200000,41.433333,18.2,48.730000,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,45.723333,17.166667,55.09,6.433333,83.423333,17.133333,41.290000,18.1,48.590000,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,45.530000,17.200000,55.09,6.366667,84.893333,17.200000,41.230000,18.1,48.590000,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


##### 2) Feature Selection

In [3]:
# dataset shape 
print("Shape before feature engineering:", raw_df.shape)

Shape before feature engineering: (19735, 28)


In [4]:
selected_cols = ["T1", 
                 "RH_1", 
                 "Appliances",
                 "lights",
                 "T_out",
                 "RH_out",
                 "Windspeed",
                 "Visibility",
                 "Press_mm_hg",
                 ]

energy_df = select_columns(raw_df, selected_cols)

##### 3) Outlier Treatment
- (missing value checking part skipped as it was previously done with the feature-engineered dataset)

In [5]:
outlier_columns = [
    "Appliances",
    "lights",
    "T_out",
    "RH_out",
    "Windspeed",
]

energy_df = clip_outliers_iqr(
    energy_df=energy_df,
    columns=outlier_columns,
    iqr_multiplier=1.5,
)

energy_df.head()

,T1,RH_1,Appliances,lights,T_out,RH_out,Windspeed,Visibility,Press_mm_hg
date,,,,,,,,,
2016-01-11 17:00:00,19.89,47.596667,60,0,6.600000,92.0,7.000000,63.000000,733.5
2016-01-11 17:10:00,19.89,46.693333,60,0,6.483333,92.0,6.666667,59.166667,733.6
2016-01-11 17:20:00,19.89,46.300000,50,0,6.366667,92.0,6.333333,55.333333,733.7
2016-01-11 17:30:00,19.89,46.066667,50,0,6.250000,92.0,6.000000,51.500000,733.8
2016-01-11 17:40:00,19.89,46.333333,60,0,6.133333,92.0,5.666667,47.666667,733.9


Outlier Treatment
- Outliers were treated using IQR-based clipping for selected columns rather than row deletion.  
- This approach preserves the time-series continuity while reducing the influence of extreme values.

#### 4) Creating time-based features

In [6]:
energy_df = create_time_features(energy_df)

energy_df[
    ["Appliances", "hour", "day_of_week", "month", "day_of_month", "is_weekend"]
].head()

,Appliances,hour,day_of_week,month,day_of_month,is_weekend
date,,,,,,
2016-01-11 17:00:00,60,17,0,1,11,0
2016-01-11 17:10:00,60,17,0,1,11,0
2016-01-11 17:20:00,50,17,0,1,11,0
2016-01-11 17:30:00,50,17,0,1,11,0
2016-01-11 17:40:00,60,17,0,1,11,0


- Time-related variables were created to capture temporal usage patterns in appliance energy consumption.  
- These include hour of the day, day of the week, month, day of the month, and a weekend indicator.

#### 5) Creating lag features

In [7]:
lag_steps = [1, 3, 6] # 10mins, 30mins, 1hr

energy_df = create_lag_features(
    energy_df=energy_df,
    target_column=TARGET_COLUMN,
    lag_steps=lag_steps,
)

energy_df[
    ["Appliances", "Appliances_lag_1", "Appliances_lag_3", "Appliances_lag_6"]
].head(10)

,Appliances,Appliances_lag_1,Appliances_lag_3,Appliances_lag_6
date,,,,
2016-01-11 17:00:00,60,NaN,NaN,NaN
2016-01-11 17:10:00,60,60.0,NaN,NaN
2016-01-11 17:20:00,50,60.0,NaN,NaN
2016-01-11 17:30:00,50,50.0,60.0,NaN
2016-01-11 17:40:00,60,50.0,60.0,NaN
2016-01-11 17:50:00,50,60.0,50.0,NaN
2016-01-11 18:00:00,60,50.0,50.0,60.0
2016-01-11 18:10:00,60,60.0,60.0,60.0
2016-01-11 18:20:00,60,60.0,50.0,50.0



- Lagged target features were created to capture temporal dependency in appliance energy consumption.  
- These features allow the model to use recent historical energy usage when predicting future consumption.

#### 6) Creating rolling features

In [8]:
window_sizes = [6, 12] # 1h rolling mean, 2h rolling mean

energy_df = create_rolling_features(
    energy_df=energy_df,
    target_column=TARGET_COLUMN,
    window_sizes=window_sizes,
)

energy_df[
    [
        "Appliances",
        "Appliances_rolling_mean_6",
        "Appliances_rolling_mean_12",
    ]
].head(15)

,Appliances,Appliances_rolling_mean_6,Appliances_rolling_mean_12
date,,,
2016-01-11 17:00:00,60,NaN,NaN
2016-01-11 17:10:00,60,NaN,NaN
2016-01-11 17:20:00,50,NaN,NaN
2016-01-11 17:30:00,50,NaN,NaN
2016-01-11 17:40:00,60,NaN,NaN
2016-01-11 17:50:00,50,55.000000,NaN
2016-01-11 18:00:00,60,55.000000,NaN
2016-01-11 18:10:00,60,55.000000,NaN
2016-01-11 18:20:00,60,56.666667,NaN


- Rolling mean features were generated to smooth short-term fluctuations and capture local energy consumption trends.  
- These features help represent recent average behavior rather than relying only on single past points.

#### 7) Creating interaction features

In [9]:
energy_df = create_interaction_features(energy_df)

interaction_columns = [
    column_name
    for column_name in energy_df.columns
    if "interaction" in column_name
]

energy_df[interaction_columns].head()

,T_out_RH_out_interaction,T1_RH_1_interaction
date,,
2016-01-11 17:00:00,607.200000,946.6977
2016-01-11 17:10:00,596.466667,928.7304
2016-01-11 17:20:00,585.733333,920.9070
2016-01-11 17:30:00,575.000000,916.2660
2016-01-11 17:40:00,564.266667,921.5700


- Interaction terms were created to capture combined effects between temperature and humidity variables.  
- These combined environmental conditions may influence appliance energy use more effectively than the original variables alone.

#### 8) Checks

In [10]:
# checking for missing values after feature eng process
energy_df.isnull().sum()[energy_df.isnull().sum() > 0]

Appliances_lag_1               1
Appliances_lag_3               3
Appliances_lag_6               6
Appliances_rolling_mean_6      5
Appliances_rolling_mean_12    11
dtype: int64

In [11]:
# removing rows with null values (caused by the lag or rolling)
energy_df = drop_feature_engineering_nulls(energy_df)

print("Shape after dropping nulls:", energy_df.shape)
energy_df.head()

Shape after dropping nulls: (19724, 21)


,T1,RH_1,Appliances,lights,T_out,RH_out,Windspeed,Visibility,Press_mm_hg,hour,day_of_week,month,day_of_month,is_weekend,Appliances_lag_1,Appliances_lag_3,Appliances_lag_6,Appliances_rolling_mean_6,Appliances_rolling_mean_12,T_out_RH_out_interaction,T1_RH_1_interaction
date,,,,,,,,,,,,,,,,,,,,,
2016-01-11 18:50:00,20.066667,46.396667,175,0,5.983333,91.166667,5.833333,40.0,734.433333,18,0,1,11,0,175.0,60.0,50.0,100.000000,77.500000,545.480556,931.026444
2016-01-11 19:00:00,20.133333,48.000000,175,0,6.000000,91.000000,6.000000,40.0,734.500000,19,0,1,11,0,175.0,70.0,60.0,119.166667,87.083333,546.000000,966.400000
2016-01-11 19:10:00,20.260000,52.726667,175,0,6.000000,90.500000,6.000000,40.0,734.616667,19,0,1,11,0,175.0,175.0,60.0,138.333333,96.666667,543.000000,1068.242267
2016-01-11 19:20:00,20.426667,55.893333,100,0,6.000000,90.000000,6.000000,40.0,734.733333,19,0,1,11,0,175.0,175.0,60.0,145.000000,100.833333,540.000000,1141.714489
2016-01-11 19:30:00,20.566667,53.893333,100,0,6.000000,89.500000,6.000000,40.0,734.850000,19,0,1,11,0,100.0,175.0,70.0,150.000000,105.000000,537.000000,1108.406222


#### 9) Saving feature-engineered dataset

In [12]:
save_dataframe(
    energy_df=energy_df,
    file_name="feature_engineered_energy_data.csv",
)